In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")
catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "340_build_gold_fact_project_scope_mix.py"
RUN_ID = str(uuid.uuid4())

FACT_PROJECT_ITEM_ESTIMATE_TABLE = f"{catalog}.gold.fact_project_item_estimate"
FACT_PROJECT_ITEM_BID_TABLE = f"{catalog}.gold.fact_project_item_bid"
TARGET_TABLE = f"{catalog}.gold.fact_project_scope_mix"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build gold.fact_project_scope_mix
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    AS

    WITH estimate_scope AS (
        SELECT
              project_id
            , project_key
            , item_work_category
            , COUNT(*) AS estimate_item_row_count
            , COUNT(DISTINCT item_specification_key) AS estimate_distinct_spec_count
            , SUM(COALESCE(extended_estimate_item_amount_calc, 0)) AS estimate_category_amount
        FROM {FACT_PROJECT_ITEM_ESTIMATE_TABLE}
        GROUP BY
              project_id
            , project_key
            , item_work_category
    )

    , estimate_project_totals AS (
        SELECT
              project_id
            , MIN(project_key) AS project_key
            , SUM(COALESCE(extended_estimate_item_amount_calc, 0)) AS estimate_item_rollup_total
            , MIN(project_engineer_total) AS min_project_engineer_total
            , MAX(project_engineer_total) AS max_project_engineer_total
            , MIN(engineer_project_total_source) AS engineer_project_total_source
        FROM {FACT_PROJECT_ITEM_ESTIMATE_TABLE}
        GROUP BY project_id
    )

    , winning_bid_scope AS (
        SELECT
              project_id
            , project_key
            , item_work_category
            , COUNT(*) AS winning_bid_item_row_count
            , COUNT(DISTINCT item_specification_key) AS winning_bid_distinct_spec_count
            , SUM(COALESCE(extended_item_amount_calc, 0)) AS winning_bid_category_amount
        FROM {FACT_PROJECT_ITEM_BID_TABLE}
        WHERE is_low_bid_project_vendor = TRUE
        GROUP BY
              project_id
            , project_key
            , item_work_category
    )

    , winning_bid_project_totals AS (
        SELECT
              project_id
            , MIN(project_key) AS project_key
            , SUM(COALESCE(extended_item_amount_calc, 0)) AS winning_bid_item_rollup_total
            , MIN(lowest_bid_amount_in_project) AS lowest_bid_amount_in_project
        FROM {FACT_PROJECT_ITEM_BID_TABLE}
        WHERE is_low_bid_project_vendor = TRUE
        GROUP BY project_id
    )

    , all_project_categories AS (
        SELECT DISTINCT
              project_id
            , project_key
            , item_work_category
        FROM {FACT_PROJECT_ITEM_ESTIMATE_TABLE}

        UNION

        SELECT DISTINCT
              project_id
            , project_key
            , item_work_category
        FROM {FACT_PROJECT_ITEM_BID_TABLE}
        WHERE is_low_bid_project_vendor = TRUE
    )

    , base AS (
        SELECT
              apc.project_id
            , apc.project_key
            , apc.item_work_category

            , COALESCE(es.estimate_item_row_count, 0) AS estimate_item_row_count
            , COALESCE(es.estimate_distinct_spec_count, 0) AS estimate_distinct_spec_count
            , COALESCE(es.estimate_category_amount, 0) AS estimate_category_amount

            , ept.estimate_item_rollup_total
            , ept.min_project_engineer_total
            , ept.max_project_engineer_total
            , ept.engineer_project_total_source

            , COALESCE(wbs.winning_bid_item_row_count, 0) AS winning_bid_item_row_count
            , COALESCE(wbs.winning_bid_distinct_spec_count, 0) AS winning_bid_distinct_spec_count
            , COALESCE(wbs.winning_bid_category_amount, 0) AS winning_bid_category_amount

            , wbpt.winning_bid_item_rollup_total
            , wbpt.lowest_bid_amount_in_project

        FROM all_project_categories apc
        LEFT JOIN estimate_scope es
            ON apc.project_id = es.project_id
           AND apc.item_work_category = es.item_work_category
        LEFT JOIN estimate_project_totals ept
            ON apc.project_id = ept.project_id
        LEFT JOIN winning_bid_scope wbs
            ON apc.project_id = wbs.project_id
           AND apc.item_work_category = wbs.item_work_category
        LEFT JOIN winning_bid_project_totals wbpt
            ON apc.project_id = wbpt.project_id
    )

    SELECT
          md5(concat_ws('|', COALESCE(project_id,''), COALESCE(item_work_category,'')))
          AS project_scope_mix_key

        , project_key
        , project_id
        , item_work_category

        , estimate_item_row_count
        , estimate_distinct_spec_count
        , estimate_category_amount
        , estimate_item_rollup_total
        , min_project_engineer_total
        , max_project_engineer_total
        , engineer_project_total_source

        , CASE
              WHEN max_project_engineer_total <> 0
              THEN estimate_category_amount / max_project_engineer_total
          END AS estimate_category_pct_of_project_total

        , CASE
              WHEN estimate_item_rollup_total <> 0
              THEN estimate_category_amount / estimate_item_rollup_total
          END AS estimate_category_pct_of_item_rollup_total

        , winning_bid_item_row_count
        , winning_bid_distinct_spec_count
        , winning_bid_category_amount
        , winning_bid_item_rollup_total
        , lowest_bid_amount_in_project

        , CASE
              WHEN lowest_bid_amount_in_project <> 0
              THEN winning_bid_category_amount / lowest_bid_amount_in_project
          END AS winning_bid_category_pct_of_low_bid_total

        , CASE
              WHEN winning_bid_item_rollup_total <> 0
              THEN winning_bid_category_amount / winning_bid_item_rollup_total
          END AS winning_bid_category_pct_of_item_rollup_total

        , winning_bid_category_amount - estimate_category_amount
          AS winning_bid_vs_estimate_category_amount_diff

        , CASE
              WHEN estimate_category_amount <> 0
              THEN winning_bid_category_amount / estimate_category_amount
          END AS winning_bid_vs_estimate_category_amount_ratio

        , (estimate_category_amount > 0 AND winning_bid_category_amount > 0)
          AS category_present_in_both_estimate_and_winning_bid

        , (estimate_category_amount > 0 AND winning_bid_category_amount = 0)
          AS category_estimate_only

        , (estimate_category_amount = 0 AND winning_bid_category_amount > 0)
          AS category_winning_bid_only

    FROM base
    """)

    row_count = spark.sql(f"""
        SELECT COUNT(*) AS row_count
        FROM {TARGET_TABLE}
    """).collect()[0]["row_count"]

    log_run("SUCCESS", row_count, "Built gold.fact_project_scope_mix")

    print("=" * 70)
    print("Built gold.fact_project_scope_mix")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise